# De Novo Mutation (DNM) Analysis Pipeline
## Combined Analysis of Parent-Offspring Trios

This notebook documents the complete pipeline for analyzing de novo mutations from two independent datasets:
- **Seplyarskiy et al., 2023** (Roulette dataset)
- **Palsson et al., 2025** (https://zenodo.org/records/14025565)

---
## Section 0: Overlap Analysis {#section0}

**STANDALONE WORKFLOW** — Compare datasets by position and position+allele.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib_venn import venn2

# ── helpers ──────────────────────────────────────────────────────────────────

def load_roulette_sets(file2_path):
    """Return (position_set, allele_set) from the Roulette corrected file."""
    positions, alleles = set(), set()
    with open(file2_path, 'r') as f:
        for line in f:
            if not line.strip():
                continue
            parts = line.strip().split()
            if len(parts) < 3:
                continue
            chrom, pos, ref_alt = parts[0], parts[1], parts[2]
            positions.add((chrom, pos))
            if '>' in ref_alt:
                ref, alt = ref_alt.split('>', 1)
                alleles.add((chrom, pos, ref, alt))
    return positions, alleles


def load_decode_sets(file1_path, skip_lines=10):
    """Return (position_set, allele_set) from the deCODE raw TSV."""
    positions, alleles = set(), set()
    with open(file1_path, 'r') as f:
        for i, line in enumerate(f, start=1):
            if i <= skip_lines or not line.strip() or line.startswith('#'):
                continue
            parts = line.strip().split()
            if len(parts) < 4:
                continue
            chrom = parts[0].replace('chr', '')
            pos, ref, alt = parts[1], parts[2], parts[3]
            positions.add((chrom, pos))
            alleles.add((chrom, pos, ref, alt))
    return positions, alleles


def compute_overlaps(decode_path, roulette_path):
    f2_pos, f2_alleles = load_roulette_sets(roulette_path)
    f1_pos, f1_alleles = load_decode_sets(decode_path)
    return {
        "f1_total_pos":    len(f1_pos),
        "f2_total_pos":    len(f2_pos),
        "overlap_pos":     len(f1_pos & f2_pos),
        "only_f1_pos":     len(f1_pos - f2_pos),
        "only_f2_pos":     len(f2_pos - f1_pos),
        "f1_total_allele": len(f1_alleles),
        "f2_total_allele": len(f2_alleles),
        "overlap_allele":  len(f1_alleles & f2_alleles),
        "only_f1_allele":  len(f1_alleles - f2_alleles),
        "only_f2_allele":  len(f2_alleles - f1_alleles),
    }


def print_stats(stats):
    def pct(n, d): return f"{100*n/d:.1f}%" if d else "N/A"
    print("\n═══════════  POSITION-ONLY OVERLAP  ═══════════")
    print(f"  deCODE total:   {stats['f1_total_pos']:,}")
    print(f"  Roulette total: {stats['f2_total_pos']:,}")
    print(f"  Overlap:        {stats['overlap_pos']:,}")
    print(f"  % of deCODE:    {pct(stats['overlap_pos'], stats['f1_total_pos'])}")
    print(f"  % of Roulette:  {pct(stats['overlap_pos'], stats['f2_total_pos'])}")
    print("\n═══════════  POSITION + ALLELE OVERLAP  ════════")
    print(f"  deCODE total:   {stats['f1_total_allele']:,}")
    print(f"  Roulette total: {stats['f2_total_allele']:,}")
    print(f"  Overlap:        {stats['overlap_allele']:,}")
    print(f"  % of deCODE:    {pct(stats['overlap_allele'], stats['f1_total_allele'])}")
    print(f"  % of Roulette:  {pct(stats['overlap_allele'], stats['f2_total_allele'])}")


def plot_venn(stats, out_png_prefix="venn_overlap"):
    COLORS = {"f1": "#1971c2", "f2": "#c2255c", "overlap": "#2f9e44",
          "bg": "white", "text": "black"}
    configs = [
        {"title":    "Position-only overlap",
         "sets":     (stats["only_f1_pos"], stats["only_f2_pos"], stats["overlap_pos"]),
         "f1_total": stats["f1_total_pos"], "f2_total": stats["f2_total_pos"],
         "f1_pct":   100*stats["overlap_pos"]/stats["f1_total_pos"] if stats["f1_total_pos"] else 0,
         "f2_pct":   100*stats["overlap_pos"]/stats["f2_total_pos"] if stats["f2_total_pos"] else 0,
         "filename": f"{out_png_prefix}_position.png"},
        {"title":    "Position + Allele overlap",
         "sets":     (stats["only_f1_allele"], stats["only_f2_allele"], stats["overlap_allele"]),
         "f1_total": stats["f1_total_allele"], "f2_total": stats["f2_total_allele"],
         "f1_pct":   100*stats["overlap_allele"]/stats["f1_total_allele"] if stats["f1_total_allele"] else 0,
         "f2_pct":   100*stats["overlap_allele"]/stats["f2_total_allele"] if stats["f2_total_allele"] else 0,
         "filename": f"{out_png_prefix}_allele.png"},
    ]
    for cfg in configs:
        fig, ax = plt.subplots(figsize=(7, 6))
        fig.patch.set_facecolor(COLORS["bg"])
        ax.set_facecolor(COLORS["bg"])
        v = venn2(subsets=cfg["sets"], set_labels=("", ""), ax=ax,
                  set_colors=(COLORS["f1"], COLORS["f2"]), alpha=0.55)
        for rid in ("10", "01", "11"):
            lbl = v.get_label_by_id(rid)
            if lbl:
                lbl.set_fontsize(13); lbl.set_fontweight("bold")
                lbl.set_color(COLORS["overlap"] if rid == "11" else COLORS["text"])
        ax.text(0.18, 0.02, f"deCODE\nn={cfg['f1_total']:,}",
            ha="center", va="center", fontsize=10, color=COLORS["f1"],
            fontweight="bold", transform=ax.transAxes)
        ax.text(0.82, 0.02, f"Roulette\nn={cfg['f2_total']:,}",
            ha="center", va="center", fontsize=10, color=COLORS["f2"],
            fontweight="bold", transform=ax.transAxes)
        ax.text(0.5, -0.06,
            f"Overlap: {cfg['sets'][2]:,}  |  {cfg['f1_pct']:.1f}% of deCODE  ·  {cfg['f2_pct']:.1f}% of Roulette",
            ha="center", va="center", fontsize=9.5, color=COLORS["overlap"],
            transform=ax.transAxes, style="italic")
        ax.set_title(cfg["title"], fontsize=13, fontweight="bold",
                     color=COLORS["text"], pad=12)
        plt.tight_layout()
        plt.savefig(cfg["filename"], dpi=180, bbox_inches="tight",
                    facecolor=fig.get_facecolor())
        print(f"Saved → {cfg['filename']}")
        plt.show()


# ── Run ───────────────────────────────────────────────────────────────────────
DECODE_RAW    = 'Decode/dnms.tsv'
ROULETTE_FILE = 'roulette_dnm_GRCh38_defined_refnuc_corrected.txt'

stats = compute_overlaps(DECODE_RAW, ROULETTE_FILE)
print_stats(stats)
plot_venn(stats, out_png_prefix="venn_overlap")

---
## Section 1: Format DNM Data Files {#section1}

Standardize mutation notation (REF>ALT) for each dataset.

### 1a: Roulette Dataset (Seplyarskiy et al., 2023) {#section1a}

Source: http://genetics.bwh.harvard.edu/downloads/Vova/Neural_net_tracks/all_mut

**Steps**: (1) Extract mutation notation (2) Get reference nucleotide (3) Correct for strand

In [ ]:
# Step 1: Convert to REF>ALT notation
input_file = 'roulette_dnm_GRCh38.txt'
output_file = 'roulette_dnm_GRCh38_defined.txt'

with open(input_file, 'r') as f:
    lines = f.readlines()

processed = []
for line in lines:
    cols = line.split()
    orig = cols[2]
    cols[2] = f"{orig[1]}>{orig[-1]}"
    processed.append(' '.join(cols))

with open(output_file, 'w') as f:
    for line in processed:
        f.write(line + '\n')

print(f'Converted {len(processed):,} Roulette records')

In [ ]:
# Step 2 & 3: Get reference nucleotide and correct strand
import subprocess
import os

def get_ref_nuc(chrom, pos, fasta):
    """Fetch reference nucleotide using samtools faidx."""
    region = f"{chrom}:{pos}-{pos}"
    try:
        result = subprocess.run(
            ['samtools', 'faidx', fasta, region],
            check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
        )
        lines = result.stdout.split('\n')
        return lines[1].strip().upper() if len(lines) >= 2 else 'N'
    except:
        return 'N'

def format_roulette_with_strand(input_file, fasta_file):
    """Add reference nucleotide and correct strand-specific calls."""
    if not os.path.exists(fasta_file):
        print('Error: Reference genome not found')
        return
    
    # Add reference nucleotides
    output_refnuc = 'roulette_dnm_GRCh38_defined_refnuc.txt'
    with open(input_file, 'r') as infile, open(output_refnuc, 'w') as outfile:
        for line in infile:
            fields = line.strip().split()
            if len(fields) < 3:
                continue
            ref_nuc = get_ref_nuc(fields[0], int(fields[1]), fasta_file)
            outfile.write('\t'.join(fields + [ref_nuc]) + '\n')
    
    # Correct strand
    comp = {'A': 'T', 'T': 'A', 'G': 'C', 'C': 'G'}
    output_corrected = 'roulette_dnm_GRCh38_defined_refnuc_corrected.txt'
    ok_lines, error_lines = [], []
    
    with open(output_refnuc, 'r') as f:
        for line in f:
            fields = line.strip().split()
            if len(fields) < 7:
                error_lines.append(line)
                continue
            chrom, pos, mutation, sid, qual, strand_info, last = fields[:7]
            ref = mutation[0]
            if ref == last:
                ok_lines.append(line)
            elif strand_info == 'D' and last in comp and comp[ref] == last:
                ref, alt = mutation.split('>')
                new_mut = f"{comp[ref]}>{comp[alt]}"
                new_line = '\t'.join([chrom, pos, new_mut] + fields[3:]) + '\n'
                ok_lines.append(new_line)
            else:
                error_lines.append(line)
    
    with open(output_corrected, 'w') as f:
        f.writelines(ok_lines)
    print(f'Strand correction: {len(ok_lines):,} OK, {len(error_lines):,} errors')

fasta_file = '/media/alexpalazzo1/ohta/Tina/SNPs_and_Indels/PrimateSNPs/Homo_sapiens.GRCh38.dna.toplevel.fa'
# format_roulette_with_strand('roulette_dnm_GRCh38_defined.txt', fasta_file)

### 1b: Palsson et al., 2025 Dataset {#section1b}

Source: https://zenodo.org/records/14025565

In [ ]:
with open('Palsson/dnms.tsv', 'r') as infile:
    lines = infile.readlines()

output_lines = []
for line in lines[10:]:  # Skip 10-line header
    if line.strip():
        cols = line.strip().split('\t')
        if len(cols) >= 4:
            ref_alt = f"{cols[2]}>{cols[3]}"
            new_line = '\t'.join([cols[0], cols[1], ref_alt] + cols[4:])
            output_lines.append(new_line)

with open('Palsson/palsson_dnm.txt', 'w') as outfile:
    outfile.write('\n'.join(output_lines))

print(f'Converted {len(output_lines):,} Palsson records')

---
## Section 2: Merge Datasets {#section2}

Combine datasets, deduplicate, and label source (roulette, palsson, or both).

In [ ]:
def load_roulette_for_merge(path):
    """Load Roulette SNPs: (chrom, pos, ref>alt)."""
    entries, seen = [], set()
    with open(path, 'r') as f:
        for line in f:
            if not line.strip() or '>' not in line:
                continue
            parts = line.strip().split()
            if len(parts) < 3:
                continue
            key = (parts[0], parts[1], parts[2])
            if key not in seen:
                seen.add(key)
                entries.append(key)
    return entries

def load_palsson_for_merge(path):
    """Load Palsson SNPs (SNPs only, no indels)."""
    entries, seen = [], set()
    with open(path, 'r') as f:
        for line in f:
            if not line.strip() or '>' not in line:
                continue
            parts = line.strip().split()
            if len(parts) < 3:
                continue
            chrom = parts[0].replace('chr', '')
            ref_alt = parts[2]
            ref, alt = ref_alt.split('>', 1)
            if len(ref) != 1 or len(alt) != 1:  # SNPs only
                continue
            key = (chrom, parts[1], ref_alt)
            if key not in seen:
                seen.add(key)
                entries.append(key)
    return entries

def merge_dnms(roulette_path, palsson_path, output_path):
    """Merge, deduplicate, and label source."""
    roulette_entries = load_roulette_for_merge(roulette_path)
    palsson_entries = load_palsson_for_merge(palsson_path)

    combined = {}
    for key in roulette_entries:
        combined[key] = 'roulette'
    for key in palsson_entries:
        combined[key] = 'both' if key in combined else 'palsson'

    with open(output_path, 'w') as out:
        out.write('chrom\tpos\tref_alt\tsource\n')
        for (chrom, pos, ref_alt), source in combined.items():
            out.write(f"{chrom}\t{pos}\t{ref_alt}\t{source}\n")

    counts = {s: sum(1 for v in combined.values() if v == s) for s in ('roulette', 'palsson', 'both')}
    print(f'Roulette-only: {counts["roulette"]:,}')
    print(f'Palsson-only:  {counts["palsson"]:,}')
    print(f'Shared (both): {counts["both"]:,}')
    print(f'Total: {len(combined):,}')

# merge_dnms('roulette_dnm_GRCh38_defined_refnuc_corrected.txt', 'Palsson/palsson_dnm.txt', 'combined_dnm.txt')

---
## Section 3: Map DNMs to TSS {#section3}

Map to relative positions within ±1 kb TSS windows.

In [ ]:
import concurrent.futures
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')

def read_tss_file(file_path):
    """Read TSS annotation: chrom, start, end, strand."""
    data = {}
    with open(file_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            chr_num = parts[0].replace('chr', '')
            start, end = int(parts[1]), int(parts[2])
            strand = parts[3]
            data.setdefault(chr_num, []).append((start, end, strand))
    return data

def process_dnm_chunk(chunk, tss_data):
    """Map DNM batch to TSS-relative positions."""
    results = []
    for line in chunk:
        try:
            parts = line.strip().split()
            chr_num = parts[0].replace('chr', '')
            coord = int(parts[1])
            extra = parts[2:]

            if chr_num in tss_data:
                for start, end, strand in tss_data[chr_num]:
                    if strand == '+' and start <= coord <= end:
                        results.append(f"{coord - start} {' '.join(extra)} {chr_num} {coord}\n")
                        break
                    elif strand == '-' and end <= coord <= start:
                        results.append(f"{start - coord} {' '.join(extra)} {chr_num} {coord}\n")
                        break
        except Exception as e:
            pass
    return results

def map_dnms(dnm_file, tss_file, output_file, chunk_size=100000):
    """Map DNMs to TSS positions."""
    tss_data = read_tss_file(tss_file)
    with open(dnm_file, 'r') as f_in, open(output_file, 'w') as f_out:
        next(f_in)  # Skip header
        with concurrent.futures.ProcessPoolExecutor() as executor:
            chunk, futures = [], []
            for line in f_in:
                chunk.append(line)
                if len(chunk) == chunk_size:
                    futures.append(executor.submit(process_dnm_chunk, chunk, tss_data))
                    chunk = []
            if chunk:
                futures.append(executor.submit(process_dnm_chunk, chunk, tss_data))
            for future in concurrent.futures.as_completed(futures):
                f_out.writelines(future.result())
    print('Mapping complete')

### 3a: Map Combined (Deduplicated) Dataset

In [ ]:
# Map the merged, deduplicated combined dataset
map_dnms(
    'combined_dnm.txt',
    '/home/alexpalazzo1/Documents/Tina/Rare_SNP/lncRNA/results/lncrna_non_testis_tss_hg38_final.txt',
    'combined_dnm_lncRNA_noexpress_mapped.txt'
)
print('✓ Combined dataset mapped')

### 3b: Map Palsson-Only Dataset (Separate Analysis)

Run as independent parallel analysis using just the Palsson et al., 2025 formatted data.

In [ ]:
# Map the Palsson-only dataset (separate analysis)
map_dnms(
    'palsson_dnm.txt',
    '/home/alexpalazzo1/Documents/Tina/Rare_SNP/lncRNA/results/lncrna_non_testis_tss_hg38_final.txt',
    'palsson_dnm_lncRNA_noexpress_mapped.txt'
)
print('✓ Palsson dataset mapped')

---
## Section 4: Filter Indels {#section4}

Safety filter: ensure only SNPs (single-nucleotide changes).

In [ ]:
def is_snp(ref_alt):
    """Check if single-nucleotide substitution (e.g., 'A>G')."""
    if '>' not in ref_alt:
        return False
    ref, alt = ref_alt.split('>', 1)
    return len(ref) == 1 and len(alt) == 1

def filter_indels(input_file, output_file):
    """Remove indels from mapped DNM file."""
    kept = skipped = 0
    with open(input_file, 'r') as f_in, open(output_file, 'w') as f_out:
        for line in f_in:
            cols = line.strip().split()
            if len(cols) < 2:
                continue
            if is_snp(cols[1]):
                f_out.write(line)
                kept += 1
            else:
                skipped += 1
    print(f'Kept: {kept:,} SNPs | Skipped: {skipped:,} indels')

---
## Section 5: Count DNMs by Position {#section5}

- Counts all 12 mutation types
- C>T split into: C>T_CpG and C>T_nonCpG
- G>A split into: G>A_CpG and G>A_nonCpG  
- Uses batched samtools queries

In [ ]:
import subprocess
import collections
import pandas as pd

FASTA_REF = '/media/alexpalazzo1/ohta/Tina/SNPs_and_Indels/PrimateSNPs/Homo_sapiens.GRCh38.dna.toplevel.fa'
file_path = 'combined_dnm_lncRNA_noexpress_mapped_noindel.txt'

mutation_types = ['A>G', 'T>C', 'C>G', 'T>G', 'C>A', 'A>T', 'G>C', 'G>T', 'C>T', 'T>A', 'A>C', 'G>A']
cpg_types = ['C>T_CpG', 'C>T_nonCpG', 'G>A_CpG', 'G>A_nonCpG']
counts = {i: {m: 0 for m in mutation_types + cpg_types} for i in range(1, 1001)}

with open(file_path, 'r') as f:
    lines = f.readlines()

# Pass 1: Parse and collect CpG context positions
parsed = []
to_fetch = collections.defaultdict(set)

for line in lines:
    parts = line.split()
    if not parts or parts[0] == 'NA':
        continue
    try:
        row_idx = int(parts[0])
    except ValueError:
        continue
    if not (1 <= row_idx <= 999) or len(parts) < 5:
        continue
    
    mut = parts[1]
    if mut not in mutation_types:
        continue
    
    chrom = parts[3]
    coord = int(parts[4])
    parsed.append((row_idx, mut, chrom, coord))
    
    if mut == 'C>T':
        to_fetch[chrom].add(coord + 1)
    elif mut == 'G>A':
        to_fetch[chrom].add(coord - 1)

print(f'Pass 1: Parsed {len(parsed):,} mutations')
print(f'CpG lookups: {sum(len(v) for v in to_fetch.values()):,} positions')

# Pass 2: Batched samtools
base_lookup = {}
for chrom, positions in sorted(to_fetch.items()):
    regions = f'/tmp/regions_{chrom}.txt'
    with open(regions, 'w') as rf:
        for pos in sorted(positions):
            rf.write(f"{chrom}:{pos}-{pos}\n")
    
    result = subprocess.run(
        ['samtools', 'faidx', FASTA_REF, '-r', regions],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
    )
    
    if result.returncode != 0:
        continue
    
    lines_out = result.stdout.strip().split('\n')
    for i in range(0, len(lines_out) - 1, 2):
        header = lines_out[i]
        seq = lines_out[i + 1].upper() if i + 1 < len(lines_out) else 'N'
        try:
            pos = int(header.split(':')[1].split('-')[0])
            base_lookup[(chrom, pos)] = seq[0] if seq else 'N'
        except:
            continue
    print(f'  chr{chrom}: fetched bases')

# Pass 3: Count with CpG context
for row_idx, mut, chrom, coord in parsed:
    counts[row_idx][mut] += 1
    
    if mut == 'C>T':
        next_base = base_lookup.get((chrom, coord + 1), 'N')
        if next_base == 'G':
            counts[row_idx]['C>T_CpG'] += 1
        else:
            counts[row_idx]['C>T_nonCpG'] += 1
    elif mut == 'G>A':
        prev_base = base_lookup.get((chrom, coord - 1), 'N')
        if prev_base == 'C':
            counts[row_idx]['G>A_CpG'] += 1
        else:
            counts[row_idx]['G>A_nonCpG'] += 1

df = pd.DataFrame.from_dict(counts, orient='index', columns=mutation_types + cpg_types)
df.to_csv('combined_dnm_lncRNA_noexpress_mapped_count.csv', index_label='Position')
print(f'✓ DNM counts saved')

---
## Section 6: Extract & Count Nucleotide Content {#section6}

**Step 1**: Convert TSS to GFF  
**Step 2**: bedtools getfasta (strand-aware)  
**Step 3**: Count nucleotides with CpG tracking

### Step 1: Convert TSS to GFF Format

In [ ]:
def tss_to_gff(input_file, output_file):
    """Convert TSS annotation to GFF format.
    
    Strips 'chr' prefix for Ensembl reference.
    BED format requires start < end regardless of strand.
    """
    written = 0
    with open(input_file, 'r') as fin, open(output_file, 'w') as fout:
        for line in fin:
            parts = line.strip().split()
            if len(parts) < 4:
                continue
            chrom = parts[0].replace('chr', '')
            a, b = int(parts[1]), int(parts[2])
            strand = parts[3]
            start, end = min(a, b), max(a, b)
            fout.write(f"{chrom}\t{start}\t{end}\t.\t.\t{strand}\n")
            written += 1
    print(f'GFF format: {written:,} regions → {output_file}')

TSS_FILE = '/home/alexpalazzo1/Documents/Tina/Rare_SNP/lncRNA/results/lncrna_non_testis_tss_hg38_final.txt'
GFF_OUT = 'fantom5_lncRNA_noexpress_1kb.gff'
# tss_to_gff(TSS_FILE, GFF_OUT)

### Step 2: Extract FASTA Sequences Using bedtools

In [ ]:
import subprocess
import os

FASTA_REF = '/media/alexpalazzo1/ohta/Tina/SNPs_and_Indels/PrimateSNPs/Homo_sapiens.GRCh38.dna.toplevel.fa'
FA_OUT = 'fantom5_lncRNA_noexpress_1kb_sequences.fa'

cmd = ['bedtools', 'getfasta', '-fi', FASTA_REF, '-bed', GFF_OUT, '-s', '-fo', FA_OUT]
# result = subprocess.run(cmd, capture_output=True, text=True)
# if result.returncode != 0:
#     raise RuntimeError('bedtools failed')

print('bedtools getfasta -s will extract strand-aware sequences')

### Step 3: Count Nucleotides per Position with CpG Tracking

In [ ]:
def calculate_nucleotide_counts(sequences_with_strands):
    """Count nucleotides at each position, tracking CpG context.
    Minus strand sequences shifted by -1 for recentering.
    """
    if not sequences_with_strands:
        return {}
    seq_len = len(sequences_with_strands[0][0])
    
    counts = {
        'A': [0] * seq_len,
        'T': [0] * seq_len,
        'G': [0] * seq_len,
        'C': [0] * seq_len,
        'C_in_CpG': [0] * seq_len,
        'G_in_CpG': [0] * seq_len,
        'C_not_in_CpG': [0] * seq_len,
        'G_not_in_CpG': [0] * seq_len,
    }
    
    for sequence, strand in sequences_with_strands:
        for i, base in enumerate(sequence):
            target_idx = i - 1 if strand == '-' else i
            if not (0 <= target_idx < seq_len):
                continue
            b = base.upper()
            
            if b == 'A':
                counts['A'][target_idx] += 1
            elif b == 'T':
                counts['T'][target_idx] += 1
            elif b == 'G':
                counts['G'][target_idx] += 1
                if i > 0 and sequence[i-1].upper() == 'C':
                    counts['G_in_CpG'][target_idx] += 1
                else:
                    counts['G_not_in_CpG'][target_idx] += 1
            elif b == 'C':
                counts['C'][target_idx] += 1
                if i < len(sequence) - 1 and sequence[i+1].upper() == 'G':
                    counts['C_in_CpG'][target_idx] += 1
                else:
                    counts['C_not_in_CpG'][target_idx] += 1
    
    return counts


def read_fasta_with_strand(fasta_path):
    """Read FASTA and detect strand from header."""
    sequences = []
    with open(fasta_path, 'r') as f:
        seq, strand = '', '+'
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith('>'):
                if seq:
                    sequences.append((seq, strand))
                seq = ''
                header = line.lower()
                strand = '-' if '(-)' in header or 'reverse' in header else '+'
            else:
                seq += line
        if seq:
            sequences.append((seq, strand))
    return sequences


def save_nucleotide_counts(counts, output_file):
    """Save nucleotide counts to TSV."""
    if not counts:
        print('ERROR: No sequences processed')
        return
    
    seq_len = len(counts['A'])
    with open(output_file, 'w') as out:
        out.write('Position\tA\tT\tC\tG\tC_in_CpG\tG_in_CpG\tC_not_in_CpG\tG_not_in_CpG\n')
        for i in range(seq_len):
            out.write(f"{i+1}\t{counts['A'][i]}\t{counts['T'][i]}\t{counts['C'][i]}\t{counts['G'][i]}\t"
                      f"{counts['C_in_CpG'][i]}\t{counts['G_in_CpG'][i]}\t{counts['C_not_in_CpG'][i]}\t"
                      f"{counts['G_not_in_CpG'][i]}\n")
    print(f'✓ Nucleotide counts: {output_file}')

print('Nucleotide counting functions defined')

---
## Section 7: Calculate Mutation Rates {#section7}

Normalize by number of mutable sites (context-aware denominators).

In [ ]:
import pandas as pd
import numpy as np

dnm_counts = pd.read_csv('combined_dnm_lncRNA_noexpress_mapped_count.csv', index_col='Position')
nuc_counts = pd.read_csv('fantom5_lncRNA_noexpress_1kb_nucleotide_counts.txt', sep='\t', index_col='Position')

dnm_counts, nuc_counts = dnm_counts.align(nuc_counts, join='inner', axis=0)

DENOM = {
    'A>G': 'A', 'A>T': 'A', 'A>C': 'A',
    'T>C': 'T', 'T>G': 'T', 'T>A': 'T',
    'C>T_nonCpG': 'C_not_in_CpG', 'C>G': 'C', 'C>A': 'C',
    'G>A_nonCpG': 'G_not_in_CpG', 'G>T': 'G', 'G>C': 'G',
}

CpG_DENOM = {
    'C>T_CpG': ('C>T_CpG', 'C_in_CpG'),
    'G>A_CpG': ('G>A_CpG', 'G_in_CpG'),
}

rates = pd.DataFrame(index=dnm_counts.index)

for mut, denom_col in DENOM.items():
    if mut in dnm_counts.columns:
        denom = nuc_counts[denom_col].replace(0, np.nan)
        rates[mut] = dnm_counts[mut] / denom

for rate_col, (mut_col, denom_col) in CpG_DENOM.items():
    if mut_col in dnm_counts.columns:
        denom = nuc_counts[denom_col].replace(0, np.nan)
        rates[rate_col] = dnm_counts[mut_col] / denom

rates.to_csv('combined_dnm_lncRNA_noexpress_mutation_rates.csv', index_label='Position')
print('✓ Mutation rates calculated')